# Train DINOv2-Style ViT-S with JAX on Kaggle TPU v5e8

Notebook này dùng cùng pipe với notebook shortcut: setup W&B secret, bật PJRT compatibility cho TPU v5e8, build/register TFDS CelebA-HQ256, clone đúng branch repo, rồi gọi `train_dinov2.py`.

`train_dinov2.py` log `dinov2/steps_per_sec`, eval deterministic theo `--eval_interval`, và sinh demo crop/patch-activity vào W&B plus `<save_dir>/demos/`. FID/generative samples không áp dụng cho DINO encoder; dùng `train.py` nếu cần FID cho DiT generator.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB2")

os.environ["WANDB_API_KEY"] = secret_value_0
os.environ["MPLBACKEND"] = "agg"
os.environ["ENABLE_PJRT_COMPATIBILITY"] = "1"
os.environ["JAX_TRACEBACK_FILTERING"] = "off"

In [ ]:
!pip install -q tfds apache_beam mlcroissant
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] += ":/root/.local/bin"

In [ ]:
!cp -r /kaggle/input/shortcut-celebahq256/tensorflow_datasets /root
%cd /kaggle/working
!git clone https://github.com/kvfrans/tfds_builders.git
%cd tfds_builders/celebahq256
!tfds build

In [ ]:
%cd /kaggle/working
!git clone https://github.com/sontungkieu/shortcut-models
%cd shortcut-models
branch = "feat/dinov2-jax-tpu"
!git checkout {branch}
!git pull
!uv sync 1>sync_out.txt 2>sync_err.txt

In [ ]:
%%bash -s "$secret_value_0"
cat > ~/.netrc <<EOF
machine api.wandb.ai login $1
EOF
chmod 600 ~/.netrc

In [ ]:
# Optional smoke test: uncomment this cell before a long run if you want to validate TPU/data/checkpoint quickly.
# %cd /kaggle/working/shortcut-models
# !git checkout {branch}
# !git pull
# !uv run train_dinov2.py \
#   --dataset_name celebahq256 \
#   --tfds_data_dir /kaggle/input/shortcut-celebahq256/tensorflow_datasets \
#   --batch_size 64 \
#   --max_steps 20 \
#   --log_interval 1 \
#   --eval_interval 10 \
#   --demo_interval 10 \
#   --save_interval 20 \
#   --save_dir /kaggle/working/ckpts/dinov2_smoke \
#   --wandb.name dinov2_smoke

In [ ]:
%cd /kaggle/working/shortcut-models
!git checkout {branch}
!git pull
!uv run train_dinov2.py \
  --dataset_name celebahq256 \
  --tfds_data_dir /kaggle/input/shortcut-celebahq256/tensorflow_datasets \
  --batch_size 64 \
  --max_steps 100000 \
  --log_interval 100 \
  --eval_interval 5000 \
  --demo_interval 5000 \
  --save_interval 5000 \
  --save_dir /kaggle/working/ckpts/dinov2_vit_s_celebahq256 \
  --wandb.name dinov2_vit_s_celebahq256 \
  --model.sharding fsdp


In [ ]:
# Resume example.
# %cd /kaggle/working/shortcut-models
# !git checkout {branch}
# !git pull
# !uv run train_dinov2.py \
#   --dataset_name celebahq256 \
#   --tfds_data_dir /kaggle/input/shortcut-celebahq256/tensorflow_datasets \
#   --batch_size 64 \
#   --max_steps 100000 \
#   --log_interval 100 \
#   --eval_interval 5000 \
#   --demo_interval 5000 \
#   --save_interval 5000 \
#   --load_dir /kaggle/working/ckpts/dinov2_vit_s_celebahq256/step_00005000 \
#   --save_dir /kaggle/working/ckpts/dinov2_vit_s_celebahq256_resume \
#   --wandb.name dinov2_vit_s_celebahq256_resume